## 1. Setup & Imports

In [2]:
import sys
sys.path.insert(0, '../src')

import logging
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import hdbscan
import umap
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from dataset import GalaxyZooDataset
from model import SimCLR

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Load Trained Model

In [3]:
# Configure paths
DATA_PATH = Path('../data/')
CHECKPOINT_PATH = Path('../src/lightning_logs/version_3/checkpoints/')  # Change version_X as needed

# Find the latest checkpoint
checkpoints = list(CHECKPOINT_PATH.glob('*.ckpt'))
if not checkpoints:
    logger.error(f'No checkpoints found in {CHECKPOINT_PATH}')
    logger.info('Available versions:')
    for v in sorted(Path('../src/lightning_logs/').glob('version_*')):
        print(f'  {v.name}')
else:
    checkpoint_path = checkpoints[0]
    logger.info(f'Loading checkpoint: {checkpoint_path}')
    
    # Load model
    model = SimCLR.load_from_checkpoint(checkpoint_path)
    model.eval()
    
    logger.info('✓ Model loaded successfully.')

INFO:__main__:Loading checkpoint: ../src/lightning_logs/version_3/checkpoints/epoch=1-step=64.ckpt
INFO:__main__:✓ Model loaded successfully.
INFO:__main__:✓ Model loaded successfully.


## 3. Extract Features from Full Dataset

In [4]:
# Calculate normalisation stats
logger.info('Computing normalisation statistics...')

csv_path = str(DATA_PATH / 'gz2spec.csv')
image_dir = str(DATA_PATH / 'images/')
mapping_csv = str(DATA_PATH / 'gz2maps.csv')

# Load data
morph_data = pd.read_csv(csv_path)
image_dir_path = Path(image_dir)
present_ids = {
    int(f.stem) for f in image_dir_path.glob("*.jpg")
}
mapping_data = pd.read_csv(mapping_csv)
mapping_data = mapping_data[mapping_data['asset_id'].isin(present_ids)]

galaxy_data = morph_data.merge(
    mapping_data,
    left_on='dr7objid',
    right_on='objid',
    how='inner'
)

# Sample 100k for analysis (or use all if fewer available)
n_samples = min(100000, len(galaxy_data))
if len(galaxy_data) > n_samples:
    galaxy_data = galaxy_data.sample(n=n_samples, random_state=42)

logger.info(f'Sampling {len(galaxy_data)} galaxies for feature extraction.')

INFO:__main__:Computing normalisation statistics...
INFO:__main__:Sampling 100000 galaxies for feature extraction.
INFO:__main__:Sampling 100000 galaxies for feature extraction.


In [5]:
# Calculate normalisation stats from subset
from PIL import Image

logger.info('Computing normalisation statistics from 10k subset...')

stat_data = galaxy_data.sample(n=min(10000, len(galaxy_data)), random_state=42)
stat_ids = stat_data['asset_id'].values

means = np.zeros(3)
stds = np.zeros(3)

for idx, galaxy_id in enumerate(stat_ids):
    if idx % 2000 == 0:
        logger.info(f'  Progress: [{idx}/{len(stat_ids)}]')
    
    image_path = image_dir_path / f"{galaxy_id}.jpg"
    if not image_path.exists():
        continue
    
    image = Image.open(image_path).convert('RGB')
    image_array = np.array(image) / 255.0
    means += image_array.mean(axis=(0, 1))
    stds += image_array.std(axis=(0, 1))

means = (means / len(stat_ids)).tolist()
stds = (stds / len(stat_ids)).tolist()

logger.info(f'Normalisation means (RGB): {[round(m, 4) for m in means]}')
logger.info(f'Normalisation stds (RGB): {[round(s, 4) for s in stds]}')

INFO:__main__:Computing normalisation statistics from 10k subset...
INFO:__main__:  Progress: [0/10000]
INFO:__main__:  Progress: [0/10000]
INFO:__main__:  Progress: [2000/10000]
INFO:__main__:  Progress: [2000/10000]


KeyboardInterrupt: 

In [ ]:
# Create dataset for feature extraction
logger.info('Creating dataset for feature extraction...')

dataset = GalaxyZooDataset(
    csv_path=csv_path,
    image_dir=image_dir,
    mapping_csv=mapping_csv,
    n_samples=len(galaxy_data),
    image_size=64,
    normalization_stats=(means, stds),
)

logger.info(f'✓ Dataset created with {len(dataset)} galaxies.')

In [ ]:
# Extract features using encoder only (discard projection head)
logger.info('Extracting features from all images...')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Custom dataloader for inference (just one view, no augmentation needed for final features)
# But we use the dataset as-is since it will augment; we could improve this later
dataloader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

features_list = []
galaxy_ids_list = []

with torch.no_grad():
    for batch_idx, (view1, view2, galaxy_id) in enumerate(dataloader):
        if batch_idx % 10 == 0:
            logger.info(f'  Batch [{batch_idx}/{len(dataloader)}]')
        
        view1 = view1.to(device)
        
        # Extract features using encoder only
        features = model(view1)
        
        features_list.append(features.cpu().numpy())
        galaxy_ids_list.extend(galaxy_id.numpy())

# Concatenate all features
X = np.vstack(features_list)
galaxy_ids = np.array(galaxy_ids_list)

logger.info(f'✓ Features extracted: shape {X.shape}')
logger.info(f'✓ Galaxy IDs collected: {len(galaxy_ids)}')

## 4. Dimensionality Reduction: PCA

In [ ]:
# Standardise features
logger.info('Standardising features...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
logger.info(f'✓ Features standardised: mean={X_scaled.mean():.4f}, std={X_scaled.std():.4f}')

In [ ]:
# PCA: Reduce to ~50 dimensions targeting 95% variance
logger.info('Fitting PCA to 95% variance...')
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

logger.info(f'✓ PCA fitted:')
logger.info(f'  Input dimensions: {X_scaled.shape[1]}')
logger.info(f'  Output dimensions: {X_pca.shape[1]}')
logger.info(f'  Explained variance: {pca.explained_variance_ratio_.sum():.4f}')

# Plot explained variance
plt.figure(figsize=(10, 5))
cumsum_var = np.cumsum(pca.explained_variance_ratio_)
plt.plot(cumsum_var, 'b-', linewidth=2)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA: Explained Variance by Component')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info('✓ PCA variance plot saved.')

## 5. Clustering: HDBSCAN

In [ ]:
# HDBSCAN clustering on PCA-reduced features
logger.info('Fitting HDBSCAN clustering...')
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, metric='euclidean')
cluster_labels = clusterer.fit_predict(X_pca)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

logger.info(f'✓ HDBSCAN clustering complete:')
logger.info(f'  Number of clusters: {n_clusters}')
logger.info(f'  Number of noise points: {n_noise} ({100*n_noise/len(cluster_labels):.1f}%)')
logger.info(f'  Cluster sizes: {np.bincount(cluster_labels[cluster_labels != -1])}')

## 6. 2D Visualisation: UMAP

In [ ]:
# UMAP projection to 2D on original features (for better structure)
logger.info('Computing UMAP projection to 2D...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = reducer.fit_transform(X_scaled)

logger.info(f'✓ UMAP projection complete: shape {X_umap.shape}')

In [ ]:
# Create result dataframe
results_df = pd.DataFrame({
    'galaxy_id': galaxy_ids,
    'cluster': cluster_labels,
    'umap_1': X_umap[:, 0],
    'umap_2': X_umap[:, 1],
})

# Merge with spectroscopic data
results_df = results_df.merge(
    morph_data,
    left_on='galaxy_id',
    right_on='dr7objid',
    how='left'
)

logger.info(f'✓ Results dataframe created: {results_df.shape}')
logger.info(f'  Columns: {list(results_df.columns)}')

In [ ]:
# Plot 1: UMAP coloured by Cluster ID
fig, ax = plt.subplots(figsize=(12, 10))

# Plot noise points first (grey)
noise_mask = results_df['cluster'] == -1
ax.scatter(
    results_df.loc[noise_mask, 'umap_1'],
    results_df.loc[noise_mask, 'umap_2'],
    c='lightgrey',
    s=10,
    alpha=0.3,
    label=f'Noise ({noise_mask.sum()})',
)

# Plot clusters
cluster_mask = results_df['cluster'] != -1
scatter = ax.scatter(
    results_df.loc[cluster_mask, 'umap_1'],
    results_df.loc[cluster_mask, 'umap_2'],
    c=results_df.loc[cluster_mask, 'cluster'],
    cmap='tab20',
    s=20,
    alpha=0.6,
)

ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Galaxy Morphology Clusters (UMAP Projection)')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Cluster ID')
ax.legend()
plt.tight_layout()
plt.savefig('../results/umap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info('✓ UMAP cluster plot saved.')

In [ ]:
# Plot 2: UMAP coloured by Galaxy Zoo vote fractions
# Try different vote fraction columns
vote_columns = [
    'smooth_fraction',
    'featured_fraction',
    'disk_fraction',
    'spiral_fraction',
]

available_cols = [col for col in vote_columns if col in results_df.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, col in enumerate(available_cols):
    ax = axes[idx]
    scatter = ax.scatter(
        results_df['umap_1'],
        results_df['umap_2'],
        c=results_df[col],
        cmap='viridis',
        s=20,
        alpha=0.6,
    )
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'Galaxy Zoo: {col.replace("_", " ").title()}')
    plt.colorbar(scatter, ax=ax)

# Hide unused subplots
for idx in range(len(available_cols), 4):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../results/umap_morphology.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info('✓ UMAP morphology plots saved.')

## 7. Validation & Summary Statistics

In [ ]:
# Cluster statistics
logger.info('\nCluster Summary Statistics:')
logger.info('='*60)

for cluster_id in sorted(results_df['cluster'].unique()):
    cluster_data = results_df[results_df['cluster'] == cluster_id]
    
    if cluster_id == -1:
        label = 'Noise'
    else:
        label = f'Cluster {cluster_id}'
    
    logger.info(f'\n{label}: {len(cluster_data)} galaxies')
    
    # Show morphology statistics
    if 'smooth_fraction' in cluster_data.columns:
        logger.info(f'  Smooth fraction: {cluster_data["smooth_fraction"].mean():.3f} ± {cluster_data["smooth_fraction"].std():.3f}')
    if 'featured_fraction' in cluster_data.columns:
        logger.info(f'  Featured fraction: {cluster_data["featured_fraction"].mean():.3f} ± {cluster_data["featured_fraction"].std():.3f}')
    if 'disk_fraction' in cluster_data.columns:
        logger.info(f'  Disk fraction: {cluster_data["disk_fraction"].mean():.3f} ± {cluster_data["disk_fraction"].std():.3f}')

logger.info('='*60)

In [ ]:
# Save results
logger.info('Saving results...')
results_df.to_csv('../results/clustering_results.csv', index=False)
logger.info('✓ Results saved to ../results/clustering_results.csv')

# Save features for later use
np.save('../results/features.npy', X)
np.save('../results/features_pca.npy', X_pca)
np.save('../results/umap_projection.npy', X_umap)
np.save('../results/galaxy_ids.npy', galaxy_ids)
logger.info('✓ Feature arrays saved.')

## Analysis Complete

Successfully completed the following steps:
1. ✓ Loaded trained SimCLR model
2. ✓ Extracted features from 100k galaxies
3. ✓ Reduced dimensionality with PCA (to ~50 dims, 95% variance)
4. ✓ Clustered with HDBSCAN (min_cluster_size=50)
5. ✓ Projected to 2D with UMAP
6. ✓ Validated clusters against Galaxy Zoo vote fractions
7. ✓ Saved results and visualisations

**Results files:**
- `../results/clustering_results.csv` - Full results with cluster IDs, UMAP coords, morphology data
- `../results/umap_clusters.png` - UMAP projection coloured by cluster ID
- `../results/umap_morphology.png` - UMAP projections coloured by vote fractions
- `../results/pca_variance.png` - PCA explained variance plot
- `../results/*.npy` - Feature arrays for further analysis